# Labeling agreement

This notebook loads the ECB and Fed labeling files for Daniel, Eric, and Sam, adds a `labeler_id` column, combines them into one dataset, and computes a few simple inter-rater agreement summaries.

In [ ]:
from itertools import combinations
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/csv")
LABEL_FILES = {
    "daniel": [
        DATA_DIR / "ecb_random_tolabel_daniel.csv",
        DATA_DIR / "fed_random_tolabel_daniel.csv",
    ],
    "eric": [
        DATA_DIR / "ecb_random_tolabel_eric.csv",
        DATA_DIR / "fed_random_tolabel_eric.csv",
    ],
    "sam": [
        DATA_DIR / "ecb_random_tolabel_sam.csv",
        DATA_DIR / "fed_random_tolabel_sam.csv",
    ],
}

pd.set_option("display.max_colwidth", 120)

In [ ]:
def detect_separator(path: Path) -> str:
    header = path.open("r", encoding="utf-8", errors="replace").readline()
    return ";" if header.count(";") > header.count(",") else ","


def load_labels(path: Path, labeler_id: str) -> pd.DataFrame:
    sep = detect_separator(path)
    df = pd.read_csv(path, sep=sep, encoding="utf-8", engine="python")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed:")].copy()

    df["labeler_id"] = labeler_id
    df["source_file"] = path.name
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    return df


frames = []
for labeler_id, paths in LABEL_FILES.items():
    for path in paths:
        frames.append(load_labels(path, labeler_id))

labels_long = pd.concat(frames, ignore_index=True).sort_values(["bank", "sentence_id", "labeler_id"])
labels_long.head()

In [ ]:
labels_long.groupby(["bank", "labeler_id"]).size().rename("rows")

In [ ]:
index_cols = [
    "bank",
    "id",
    "sentence_id",
    "date",
    "chair",
    "text",
]

labels_wide = (
    labels_long[index_cols + ["labeler_id", "label"]]
    .drop_duplicates(subset=["sentence_id", "labeler_id"])
    .pivot(index=index_cols, columns="labeler_id", values="label")
    .reset_index()
)

labels_wide.columns.name = None
labels_wide.head()

In [ ]:
labeler_cols = ["daniel", "eric", "sam"]
coverage_summary = pd.DataFrame(
    {
        "non_missing_labels": labels_wide[labeler_cols].notna().sum(axis=1)
    }
).value_counts().rename("sentences")

coverage_summary

In [ ]:
complete_cases = labels_wide.dropna(subset=labeler_cols).copy()
complete_cases["all_three_match"] = complete_cases[labeler_cols].nunique(axis=1) == 1

overall_exact_agreement = complete_cases["all_three_match"].mean()
agreement_by_bank = complete_cases.groupby("bank")["all_three_match"].mean().rename("exact_agreement")

print(f"Complete cases with all three labelers: {len(complete_cases)}")
print(f"Overall exact agreement (all three match): {overall_exact_agreement:.3f}")
agreement_by_bank

In [ ]:
def cohens_kappa(left: pd.Series, right: pd.Series) -> float:
    paired = pd.DataFrame({"left": left, "right": right}).dropna()
    if paired.empty:
        return float("nan")

    agreement = (paired["left"] == paired["right"]).mean()
    confusion = pd.crosstab(paired["left"], paired["right"])
    row_probs = confusion.sum(axis=1) / confusion.to_numpy().sum()
    col_probs = confusion.sum(axis=0) / confusion.to_numpy().sum()
    expected = row_probs.mul(col_probs, fill_value=0).sum()
    if expected == 1:
        return float("nan")
    return (agreement - expected) / (1 - expected)


def pairwise_metrics(df: pd.DataFrame, group_cols: list[str] | None = None) -> pd.DataFrame:
    rows = []
    groups = [("overall", df)] if not group_cols else df.groupby(group_cols, dropna=False)

    for group_name, group_df in groups:
        for left, right in combinations(labeler_cols, 2):
            pair_df = group_df.dropna(subset=[left, right])
            if pair_df.empty:
                continue

            rows.append(
                {
                    "group": group_name,
                    "labeler_a": left,
                    "labeler_b": right,
                    "n": len(pair_df),
                    "percent_agreement": (pair_df[left] == pair_df[right]).mean(),
                    "cohens_kappa": cohens_kappa(pair_df[left], pair_df[right]),
                }
            )

    return pd.DataFrame(rows)


pairwise_overall = pairwise_metrics(labels_wide)
pairwise_by_bank = pairwise_metrics(labels_wide, group_cols=["bank"])

pairwise_overall

In [ ]:
pairwise_by_bank

In [ ]:
disagreements = complete_cases.loc[~complete_cases["all_three_match"], ["bank", "sentence_id", "daniel", "eric", "sam", "text"]]
disagreements.head(20)

If you want a single export for later analysis, you can save `labels_long` or `labels_wide` with `to_csv(...)` in a new cell.